# TP 1 - Prompt Engineering

Ce notebook introduit les bases de l'interaction avec un LLM via l'API Google GenAI.

On construit progressivement un assistant de voyage, du prompt brut jusqu'à la sortie structurée.


### 0.1. Documentation générale des librairies utilisées

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

### 0.2. Le format Notebook (.ipynb)

Un fichier `.ipynb` est un document interactif composé de **cellules** que l'on exécute une à une, dans l'ordre.

Il existe deux types de cellules :
- **Cellule code** : contient du code Python exécutable
- **Cellule markdown** : contient du texte formaté (titres, listes, liens…)

Exécutez chaque cellule avec `Shift+Enter` ou le bouton ▶ dans la barre d'outils.

In [ ]:
import json

from google.genai import types
from pydantic import BaseModel

from shared.config import ROOT_DIR, genai_client, project_settings
from shared.llm_utils import LLMRequest, LLMResponse
from shared.misc_utils import write_json_file

LOG_DIR = ROOT_DIR / "TP1_travel_planner_LLM" / "logs"

### 0.3. Use case principal
Votre objectif est de répondre à une question

In [ ]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire de voyage. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

### 0.4. Récapitulatif des fonctions utilisées dans ce notebook

**Fonctions et classes à utiliser**

- `project_settings` : objet qui centralise la configuration partagée du modèle (température, top_p, top_k, max_tokens)
- `genai_client` : client qui s'authentifie à l'API Google GenAI, créé une seule fois

--> Disponibles dans `shared/config.py`

- `LLMRequest` : classe qui représente les données d'entrée d'un appel LLM (`user_prompt` obligatoire, `system_prompt` optionnel)
- `LLMResponse` : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)

--> Disponibles dans `shared/llm_utils.py`

**Fonctions à coder dans ce notebook**

- `run_llm` : génère un `LLMResponse` à partir d'un texte en entrée
- `run_llm_structured` : comme `run_llm`, mais impose une sortie JSON conforme à un `response_schema` (voir partie 3)

--> À implémenter ici, puis à copier dans `shared/llm_utils.py` une fois que les fonctions marchent

**(Optionnel)** vous pouvez aussi coder les fonctions équivalentes pour lancer une **inférence LLM en local** :
- `run_llm_local`
- `run_llm_structured_local`

#### Fonctions auxiliaires pour calculer le coût et afficher l'usage des tokens

In [ ]:
def token_estimate_cost_usd(
    token_usage: dict[str, int],
    input_price_per_1m_tokens_usd: float = 0.30,
    output_price_per_1m_tokens_usd: float = 2.50,
) -> float:
    input_cost = token_usage["input_tokens"] * input_price_per_1m_tokens_usd / 1_000_000
    output_cost = token_usage["output_tokens"] * output_price_per_1m_tokens_usd / 1_000_000
    return input_cost + output_cost


def token_print_report(token_usage: dict[str, int]) -> None:
    estimated_cost = token_estimate_cost_usd(token_usage)
    print(
        f"tokens : entrée={token_usage['input_tokens']} | "
        f"sortie={token_usage['output_tokens']} | "
        f"total={token_usage['total_tokens']}"
    )
    print(f"coût estimé (USD) : {estimated_cost:.6f}")


### 0.5. Coder la fonction "run_llm"

Cette fonction sert de base pour tous les TPs, elle permet d'appeler l'API Gemini avec une requête textuelle.

Pour l'instant, nous laisserons le `system_prompt` en `None`

In [ ]:
async def run_llm(request: LLMRequest) -> LLMResponse:
    """Appelle le LLM avec un prompt utilisateur et un prompt système, et retourne la réponse du modèle ainsi que les métriques d'usage des tokens

    Entree
    - request : LLMRequest (system_prompt optionnel, user_prompt obligatoire)

    Sortie
    - LLMResponse avec texte, metriques tokens et reponse brute
    """
    # DOC (GenerationConfig) : https://ai.google.dev/api/generate-content#GenerationConfig
    # DOC (thinking_budget) : https://ai.google.dev/gemini-api/docs/generate-content/thinking
    config = types.GenerateContentConfig(
        temperature=project_settings.llm_temperature,
        top_p=project_settings.llm_top_p,
        top_k=project_settings.llm_top_k,
        max_output_tokens=project_settings.llm_max_output_tokens,
        thinking_config=types.ThinkingConfig(
            thinking_budget=project_settings.llm_thinking_budget
        ),
        system_instruction=request.system_prompt,
    )

    # TODO : appeler le modele et retourner la reponse (LLMResponse)
    # DOC (generate_content) : https://github.com/googleapis/python-genai#generate-content-asynchronous-non-streaming
    response = await ...

    usage_metadata = response.usage_metadata

    return ...

---
## 1. Prompt simple

Sans system prompt, le modèle s'appuie uniquement sur son instruction générale de base et choisit librement le style et la structure de sa réponse.

**Attention à bien séparer la requête utilisateur et les tâches du LLM.**

Ici, par exemple :
- La requête utilisateur porte sur une ville, des envies et des contraintes spécifiques
- Le system prompt décrit le rôle du modèle (agent de voyage) et la forme attendue de la réponse (structure, style, format).

--> Le system prompt est réutilisable : **il ne dépend pas d'un cas d'usage précis**.

Il faut donc trouver un bon équilibre entre prompt généraliste (réutilisable) et prompt spécifique (adapté à la requête utilisateur)

Dans ce cas, le prompt système doit définir le **rôle d'agent de voyage**, poser des contraintes de contenu et de **style**, imposer une **structure** de réponse claire, et vérifier que les **contraintes utilisateur** (budget, temps) sont valides.

### 1.1. Construire la requête et appeler le LLM

In [ ]:
request_v1 = LLMRequest(
    system_prompt=...,
    user_prompt=...,
)
run_result_v1 = ...
final_text_v1 = ...

token_usage_v1 = {
    "input_tokens": ...,
    "output_tokens": ...,
    "total_tokens": ...,
}

log_path_v1 = LOG_DIR / "llm_output_v1.txt"
write_json_file(file_path=log_path_v1, data=run_result_v1.raw_response)

print(final_text_v1)

### 1.2. Afficher l'usage des tokens

In [ ]:
token_print_report(token_usage_v1)


---
## 2. System prompt structuré

Sans system prompt, le modèle choisit librement sa structure de réponse.

--> Un system prompt permet d'imposer un rôle, des contraintes et un format de sortie cohérent.

Pour améliorer la réponse, définir dans `system_prompt_v2` :
- un **rôle** explicite pour le modèle
- des **contraintes** de contenu et de style
- une **structure** en **4 sections** : résumé, itinéraire, budget, conseils

### 2.1. Rédiger le system prompt

In [ ]:
system_prompt_v2 = """
## Instructions
Tu es ...

Contraintes à respecter :
- ...

Structure de réponse :
1) ...

...
"""

### 2.2. Construire la requête et appeler le LLM

In [ ]:
request_v2 = ...
run_result_v2 = ...
final_text_v2 = ...

token_usage_v2 = {
    "input_tokens": int(run_result_v2.input_tokens),
    "output_tokens": int(run_result_v2.output_tokens),
    "total_tokens": int(run_result_v2.total_tokens),
}

log_path_v2 = LOG_DIR / "llm_output_v2.txt"
write_json_file(file_path=log_path_v2, data=run_result_v2.raw_response)

print(final_text_v2)

### 2.3. Afficher l'usage des tokens

In [ ]:
token_print_report(token_usage_v2)


---
## 3. Sortie structurée (Structured Output)

On pourrait demander dans le prompt au LLM de suivre un format JSON (« réponds avec un JSON comme ceci : ... »), ce qui est fait souvent en pratique, mais reste peu fiable et complexifie le parsing.

Gemini propose une solution plus propre : la **sortie structurée** (*structured output*). Avec `response_schema` (un modèle Pydantic) et `response_mime_type="application/json"`, l'API garantit un JSON valide conforme au schéma, plus besoin de parsing.

Doc : https://ai.google.dev/gemini-api/docs/structured-output

### 3.1. Coder la fonction de sortie structurée

Même logique que pour l'appel simple, avec un schéma de sortie en plus.

In [ ]:
async def run_llm_structured(request: LLMRequest, response_schema: type[BaseModel]) -> LLMResponse:
    """Executer un appel LLM avec sortie structuree (JSON garanti conforme a response_schema)

    Entree
    - request : LLMRequest (system_prompt optionnel, user_prompt obligatoire)
    - response_schema : modele Pydantic decrivant le JSON attendu

    Sortie
    - LLMResponse avec texte (JSON valide), metriques tokens et reponse brute
    """
    # DOC (response_schema) : https://ai.google.dev/gemini-api/docs/structured-output
    config = types.GenerateContentConfig(
        temperature=project_settings.llm_temperature,
        top_p=project_settings.llm_top_p,
        top_k=project_settings.llm_top_k,
        max_output_tokens=project_settings.llm_max_output_tokens,
        thinking_config=types.ThinkingConfig(
            thinking_budget=project_settings.llm_thinking_budget
        ),
        system_instruction=request.system_prompt,
        response_mime_type=...,
        response_schema=...,
    )

    # TODO : appeler le modele et retourner la reponse (LLMResponse)
    # DOC (generate_content) : https://github.com/googleapis/python-genai#generate-content-asynchronous-non-streaming
    response = await ...

    usage_metadata = response.usage_metadata

    return ...

### 3.2. Définir la structure de sortie

Définir un modèle Pydantic décrivant l'agenda (`DayPlan` libre : à vous de choisir comment représenter repas/activités).

In [ ]:
# DOC (response_schema) : https://ai.google.dev/gemini-api/docs/structured-output
class ActivityItem(BaseModel):
    estimated_cost_eur: float  # nécessaire pour la vérification budget (partie suivante)
    # ajouter les champs qui vous semblent utiles
    # Exemples : titre, description, adresse, horaire, ...


class MealItem(BaseModel):
    estimated_cost_eur: float  # nécessaire pour la vérification budget (partie suivante)
    # ajouter les champs qui vous semblent utiles
    # Exemples : nom, adresse, horaire, type, ...
    ...


class DayPlan(BaseModel):
    # Remplir ici avec les activités et les repas
    # Exemple : repas midi / soir et activité matin / après-midi
    ...


class TravelAgenda(BaseModel):
    agenda: list[DayPlan]

### 3.3. Construire la requête et appeler le LLM

In [ ]:
request_v3 = ...
run_result_v3 = ...
final_text_v3 = ...

token_usage_v3 = {
    "input_tokens": int(run_result_v3.input_tokens),
    "output_tokens": int(run_result_v3.output_tokens),
    "total_tokens": int(run_result_v3.total_tokens),
}

log_path_v3 = LOG_DIR / "llm_output_v3.txt"
write_json_file(file_path=log_path_v3, data=run_result_v3.raw_response)

print(final_text_v3)

### 3.4. Afficher l'usage des tokens

In [ ]:
token_print_report(token_usage_v3)


### 3.5. Parser le JSON

`response_schema` garantit que `final_text_v3` est déjà un JSON valide conforme au schéma : plus besoin de fonction de parsing maison, un simple `json.loads` suffit.

In [ ]:
parsed_json_v3 = ...
print(json.dumps(obj=parsed_json_v3, ensure_ascii=False, indent=2))

### 3.6. Vérifier le budget

Finalement, on peut traiter le JSON parsé pour calculer le coût total estimé des activités proposées, et vérifier que cela respecte la contrainte de budget donnée dans la requête utilisateur.

In [ ]:
...

total_cost_eur = ...
average_per_day = ...

print(f"Total estimé : {total_cost_eur:.2f} EUR")
print(f"Moyenne par jour : {average_per_day:.2f} EUR")

---
## Déplacer vers shared/

Les fonctions codées dans ce notebook sont utilisées dans les TPs suivants.

Copiez-les dans `shared/` avant de passer à la suite :
- `run_llm`
- `run_llm_structured`

--> À copier dans `shared/llm_utils.py`